# Lab 7 - Visualisation Météo (Bokeh)

In [2]:
import sys
print(sys.executable)

c:\Users\moham\AppData\Local\Programs\Python\Python310\python.exe


In [7]:
import pandas as pd
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, HoverTool, DateRangeSlider
from bokeh.layouts import column
from bokeh.io import output_notebook
from bokeh.transform import factor_cmap
import numpy as np

output_notebook()

# ✅ Ton chemin absolu (corrigé)
df = pd.read_csv(r'C:\Users\moham\OneDrive\Documents\ESAIP\ING4\visualisation\Data_Visualization_Course_2026\Interactive-Data-Visualization-with-Python\datasets\chap5_data\AirPassengersDates.csv')

# 🔍 Voir les colonnes
print(df.columns)

# ✅ Adapter au dataset AirPassengers
df.rename(columns={
    'Month': 'Date',
    '#Passengers': 'Temperature'
}, inplace=True)

# ✅ Conversion des types
df['Date'] = pd.to_datetime(df['Date'])
df['Temperature'] = df['Temperature'].astype(float)

# ✅ Trier
df = df.sort_values('Date')

df.head()

Loading BokehJS ...

Index(['Date', '#Passengers'], dtype='object')


,Date,Temperature
0,1949-01-12,112.0
1,1949-02-24,118.0
2,1949-03-22,132.0
3,1949-04-05,129.0
4,1949-05-24,121.0


## Question 1

In [8]:

source = ColumnDataSource(df)

p = figure(x_axis_type='datetime', title="Daily Minimum Temperatures",
           tools="pan,wheel_zoom,reset")

p.line('Date','Temperature', source=source)

p.xaxis.axis_label = "Date"
p.yaxis.axis_label = "Temperature (°C)"

hover = HoverTool(tooltips=[("Date","@Date{%F}"),("Temp","@Temperature")],
                  formatters={'@Date':'datetime'})
p.add_tools(hover)

show(p)


## Question 2

In [9]:

df['Rolling_Avg'] = df['Temperature'].rolling(30).mean()

source = ColumnDataSource(df)

p = figure(x_axis_type='datetime', title="Temp + Rolling Avg",
           tools="pan,wheel_zoom,reset")

p.line('Date','Temperature', source=source, color="blue", legend_label="Temp")
p.line('Date','Rolling_Avg', source=source, color="red", legend_label="Rolling Avg")

hover = HoverTool(tooltips=[
    ("Date","@Date{%F}"),
    ("Temp","@Temperature"),
    ("Avg","@Rolling_Avg")
], formatters={'@Date':'datetime'})
p.add_tools(hover)

p.legend.click_policy="hide"

show(p)


## Question 3

In [10]:

df['Month'] = df['Date'].dt.month_name()

groups = df.groupby('Month')['Temperature']

months = list(groups.groups.keys())

q1 = groups.quantile(0.25)
q2 = groups.quantile(0.5)
q3 = groups.quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5*iqr
lower = q1 - 1.5*iqr

p = figure(x_range=months, title="Boxplot Mensuel")

p.segment(months, upper, months, q3)
p.segment(months, lower, months, q1)

p.vbar(months, 0.7, q2, q3)
p.vbar(months, 0.7, q1, q2)

show(p)


## Question 4

In [11]:

df['Year'] = df['Date'].dt.year.astype(str)

groups = df.groupby('Year')['Temperature']

years = list(groups.groups.keys())
medians = groups.median()

source = ColumnDataSource(dict(year=years, median=medians))

p = figure(x_range=years, title="Boxplot Annuel")

colors = ["#%02x%02x%02x" % (int(m*5)%255,100,150) for m in medians]

p.vbar(x=years, top=medians, width=0.9, color=colors)

show(p)


## Question 5

In [12]:

source = ColumnDataSource(df)

p = figure(x_axis_type='datetime', height=300)
line = p.line('Date','Temperature', source=source)

slider = DateRangeSlider(start=df['Date'].min(), end=df['Date'].max(),
                         value=(df['Date'].min(), df['Date'].max()))

def update(attr, old, new):
    start, end = slider.value_as_datetime
    new_df = df[(df['Date']>=start)&(df['Date']<=end)]
    source.data = ColumnDataSource(new_df).data

slider.on_change('value', update)

show(column(slider, p))


You are generating standalone HTML/JS output, but trying to use real Python
callbacks (i.e. with on_change or on_event). This combination cannot work.

Only JavaScript callbacks may be used with standalone output. For more
information on JavaScript callbacks with Bokeh, see:

    https://docs.bokeh.org/en/latest/docs/user_guide/interaction/js_callbacks.html

Alternatively, to use real Python callbacks, a Bokeh server application may
be used. For more information on building and running Bokeh applications, see:

    https://docs.bokeh.org/en/latest/docs/user_guide/server.html



## Question 6

In [14]:
# ✅ On sélectionne uniquement la colonne numérique
monthly = df.set_index('Date')[['Temperature']].resample('M').mean()

# Trend
monthly['Trend'] = monthly['Temperature'].rolling(3).mean()

# Seasonality
monthly['Seasonal'] = monthly['Temperature'] - monthly['Trend']

# ⚠️ Important pour Bokeh → remettre Date en colonne
monthly = monthly.reset_index()

source = ColumnDataSource(monthly)

p1 = figure(x_axis_type='datetime', title="Monthly")
p1.line('Date','Temperature', source=source)

p2 = figure(x_axis_type='datetime', title="Trend", x_range=p1.x_range)
p2.line('Date','Trend', source=source)

p3 = figure(x_axis_type='datetime', title="Seasonality", x_range=p1.x_range)
p3.line('Date','Seasonal', source=source)

show(column(p1,p2,p3))

C:\Users\moham\AppData\Local\Temp\ipykernel_26628\4237012370.py:2: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly = df.set_index('Date')[['Temperature']].resample('M').mean()
